In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

from datetime import datetime

Load Silver Delta Table

In [0]:
df = spark.table(
    "customer_transactions_silver"
)

display(df.limit(5))

customer_id,customer_name,customer_age,customer_gender,customer_segment,loyalty_tier,country,state,city,region,product_id,product_name,product_category,subcategory,brand,transaction_id,order_date,shipment_date,payment_method,sales_channel,currency,quantity,unit_price,discount,tax,shipping_cost,total_sales,profit,profit_margin,warehouse,delivery_days,supplier,inventory_level,return_flag,website_visits,cart_size,coupon_used,session_duration,load_date
CUST114305,Haruto Martinez,null,Female,Corporate,Bronze,USA,New York,Albany,North America,PRD21153,AutoPro Accessories 975,Automotive,Accessories,AutoPro,TXN31000000,2026-03-24T05:11:00Z,2026-04-03T05:11:00Z,Gift Card,Online,USD,3,515.78,0.276,176.34,5.550999999999999,1302.17,133.07,0.1022,WH-NA-02,10.0,Pinnacle Supply Co,null,0,4,6,1.0,15.1,2026-01-03
CUST112811,Lucas Allen,31.0,Male,Corporate,Bronze,USA,New York,Buffalo,North America,PRD20728,Rally Footwear 807,Sports & Outdoors,Footwear,Rally,TXN31000001,2026-03-21T19:32:00Z,2026-03-29T19:32:00Z,Debit Card,In-Store,USD,6,308.46,0.177,null,4.9399999999999995,1709.47,464.54,0.2717,WH-NA-02,8.0,Vertex Distribution,128.0,0,3,9,1.0,4.7,2026-01-03
CUST113276,Li Anderson,41.0,Female,Corporate,Silver,USA,California,San Diego,North America,PRD20216,PureGlow Skincare 133,Beauty & Personal Care,Skincare,PureGlow,TXN31000002,2026-03-13T05:42:00Z,2026-03-21T05:42:00Z,Gift Card,In-Store,USD,6,19.26,0.269,11.4,10.894000000000002,106.77,36.17,0.3388,WH-NA-02,8.0,QuickChain Imports,78.0,1,9,6,null,10.4,2026-01-03
CUST113214,Richard Jones,40.0,Male,Small Business,Bronze,Mexico,Jalisco,Guadalajara,Latin America,PRD20182,TorqueTech Car Care 444,Automotive,Car Care,TorqueTech,TXN31000003,2026-03-02T19:53:00Z,2026-03-10T19:53:00Z,Debit Card,Mobile App,MXN,4,311.9,0.434,40.16,null,760.77,-89.83,null,WH-EU-01,null,QuickChain Imports,113.0,0,8,4,1.0,10.6,2026-01-03
CUST113956,Ava Wilson,21.0,Male,Consumer,Platinum,Vietnam,Hanoi,Hanoi City,Asia Pacific,PRD20632,Domus Decor 984,Home & Kitchen,Decor,Domus,TXN31000004,2026-03-14T09:46:00Z,2026-03-18T09:46:00Z,Credit Card,Online,VND,4,395.65,0.033,221.72,5.707,1757.8,572.9,null,WH-EU-02,4.0,Meridian Traders,188.0,0,8,6,0.0,3.7,2026-01-03


In [0]:
total_records = df.count()

print("Total Records:", total_records)

Total Records: 300000


Duplicate Transaction Detection

In [0]:
transaction_duplicates = (
    df.groupBy("transaction_id")
      .count()
      .filter(F.col("count") > 1)
)

duplicate_count = transaction_duplicates.count()

duplicate_percentage = (
    duplicate_count / total_records
) * 100


print("Duplicate transactions:", duplicate_count)
print("Duplicate %:", duplicate_percentage)

Duplicate transactions: 0
Duplicate %: 0.0


customer + transaction date duplicates

In [0]:
customer_date_duplicates = (
    df.groupBy(
        "customer_id",
        "order_date"
    )
    .count()
    .filter(F.col("count") > 1)
)

customer_duplicate_count = (
    customer_date_duplicates.count()
)

Null Validation

In [0]:
columns_to_check = df.columns

null_results = []

for column in columns_to_check:

    null_count = (
        df.filter(
            F.col(column).isNull()
        )
        .count()
    )

    null_percentage = (
        null_count / total_records
    ) * 100


    status = (
        "PASS"
        if null_percentage == 0
        else "FAIL"
    )


    null_results.append(
        (
            column,
            null_count,
            null_percentage,
            status
        )
    )

In [0]:
null_df = spark.createDataFrame(
    null_results,
    [
        "column_name",
        "null_count",
        "null_percentage",
        "quality_status"
    ]
)

display(null_df)

column_name,null_count,null_percentage,quality_status
customer_id,0,0.0,PASS
customer_name,0,0.0,PASS
customer_age,14634,4.878,FAIL
customer_gender,11314,3.7713333333333336,FAIL
customer_segment,0,0.0,PASS
loyalty_tier,0,0.0,PASS
country,0,0.0,PASS
state,0,0.0,PASS
city,0,0.0,PASS
region,0,0.0,PASS


Schema Validation

In [0]:
expected_schema = {

"customer_id":"string",
"customer_name":"string",
"customer_age":"int",
"product_id":"string",
"transaction_id":"string",
"quantity":"int",
"unit_price":"double",
"total_sales":"double",
"profit":"double",
"profit_margin":"double"

}
actual_schema = dict(
    df.dtypes
)

Missing columns

In [0]:
missing_columns = set(
    expected_schema.keys()
) - set(
    actual_schema.keys()
)

missing_columns

set()

Unexpected columns

In [0]:
unexpected_columns = set(
    actual_schema.keys()
) - set(
    expected_schema.keys()
)

unexpected_columns

{'brand',
 'cart_size',
 'city',
 'country',
 'coupon_used',
 'currency',
 'customer_gender',
 'customer_segment',
 'delivery_days',
 'discount',
 'inventory_level',
 'load_date',
 'loyalty_tier',
 'order_date',
 'payment_method',
 'product_category',
 'product_name',
 'region',
 'return_flag',
 'sales_channel',
 'session_duration',
 'shipment_date',
 'shipping_cost',
 'state',
 'subcategory',
 'supplier',
 'tax',
 'warehouse',
 'website_visits'}

Data type mismatch

In [0]:
type_errors = []

for col, expected_type in expected_schema.items():

    if col in actual_schema:

        if actual_schema[col] != expected_type:

            type_errors.append(
                (
                    col,
                    expected_type,
                    actual_schema[col]
                )
            )


type_errors

[('customer_age', 'int', 'double')]

**Business Rule Validation**

Rule 1 - quantity > 0

In [0]:
quantity_fail = (
    df.filter(
        F.col("quantity") <= 0
    )
    .count()
)

Rule 2 - unit_price > 0

In [0]:
unit_price_fail = (
    df.filter(
        F.col("unit_price") <= 0
    )
    .count()
)

Rule 3 - total_sales > 0

In [0]:
sales_fail = (
    df.filter(
        F.col("total_sales") <= 0
    )
    .count()
)

Rule 4 - profit_margin between 0 and 1

In [0]:
profit_margin_fail = (
    df.filter(
        (F.col("profit_margin") < 0) |
        (F.col("profit_margin") > 1)
    )
    .count()
)

Rule 5 - delivery_days >= 0

In [0]:
delivery_fail = (
    df.filter(
        F.col("delivery_days") < 0
    )
    .count()
)

Rule 6 - discount between 0 and 1

In [0]:
discount_fail = (
    df.filter(
        (F.col("discount") < 0) |
        (F.col("discount") > 1)
    )
    .count()
)

Rule 7 - return_flag only 0 or 1

In [0]:
return_flag_fail = (
    df.filter(
        ~F.col("return_flag").isin(0,1)
    )
    .count()
)

**Data Quality Score**

In [0]:
total_nulls = (
    null_df
    .agg(
        F.sum("null_count")
    )
    .collect()[0][0]
)


completeness_score = (
    100 -
    ((total_nulls /
    (total_records * len(df.columns)))
    * 100)
)

In [0]:
uniqueness_score = (
    100 -
    duplicate_percentage
)

In [0]:
total_rule_failures = (
    quantity_fail +
    unit_price_fail +
    sales_fail +
    profit_margin_fail +
    delivery_fail +
    discount_fail +
    return_flag_fail
)


validity_score = (
    100 -
    ((total_rule_failures /
    total_records)
    *100)
)

In [0]:
schema_score = (
    100
    if (
        len(missing_columns)==0
        and len(unexpected_columns)==0
        and len(type_errors)==0
    )
    else 0
)

In [0]:
data_quality_score = (
    completeness_score +
    uniqueness_score +
    validity_score +
    schema_score
) / 4


print(
    "Data Quality Score:",
    data_quality_score
)

Data Quality Score: 72.92801495726496


**Create Quality Results Delta Table**

In [0]:
run_date = datetime.now()


results = [

(
"customer_transactions",
"duplicate_transaction_check",
"transaction_id",
duplicate_count,
total_records,
duplicate_percentage,
data_quality_score,
"PASS" if duplicate_count==0 else "FAIL"

),

(
"customer_transactions",
"null_validation",
"all_columns",
total_nulls,
total_records,
(total_nulls/total_records)*100,
data_quality_score,
"PASS" if total_nulls==0 else "FAIL"

)

]

In [0]:
quality_df = spark.createDataFrame(
    results,
[
"dataset_name",
"rule_name",
"column_name",
"failed_records",
"total_records",
"failure_percentage",
"quality_score",
"status"
]
)

In [0]:
quality_df = (
    quality_df
    .withColumn(
        "run_date",
        F.current_timestamp()
    )
)

In [0]:
quality_path = (
"abfss://metadata@<your_storage_account>.dfs.core.windows.net/"
"quality_results/"
)

In [0]:
(
quality_df.write
.format("delta")
.mode("append")
.save(quality_path)
)

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS quality_results_delta
USING DELTA
LOCATION 
'abfss://metadata@<your_storage_account>.dfs.core.windows.net/quality_results/'
""")

DataFrame[]

- Test - bad table


In [0]:
from pyspark.sql.functions import when, rand

bad_df = (
    df.withColumn(
        "quantity",
        when(
            rand() < 0.10,
            F.lit(-10)
        )
        .otherwise(F.col("quantity"))
    )
)

In [0]:
bad_df.filter(
F.col("quantity")<=0
).count()

29963

In [0]:
print("Original:", df.count())
print("Bad DF:", bad_df.count())

Original: 300000
Bad DF: 300000


In [0]:
bad_df.groupBy("quantity").count().show()

+--------+-----+
|quantity|count|
+--------+-----+
|     -10|29963|
|      12|   59|
|       1|20527|
|      13|   18|
|       6|21597|
|       3|64439|
|       5|38542|
|      15|    1|
|       9| 1702|
|       4|56577|
|       8| 4592|
|       7|10593|
|      10|  657|
|      11|  195|
|      14|    2|
|       2|50536|
+--------+-----+

